In [1]:
import pandas as pd
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

data_folder = project_folder / "data"
feature_file = data_folder / "propertylens_featured.csv"

df = pd.read_csv(feature_file)

# Create market segments from the existing price bands
segment_map = {
    "Below 50L": "Budget",
    "50L-1Cr": "Budget",
    "1Cr-2Cr": "Mid-Market",
    "2Cr-5Cr": "Premium",
    "5Cr-10Cr": "High-End",
    "Above 10Cr": "Luxury"
}

df["market_segment"] = df["price_band"].map(segment_map)

segment_order = [
    "Budget",
    "Mid-Market",
    "Premium",
    "High-End",
    "Luxury"
]

# Overall segment summary
segment_summary = (
    df.groupby("market_segment", observed=True)
    .agg(
        listings=("price_lakh", "size"),
        median_price_lakh=("price_lakh", "median"),
        median_area_sqft=("area_sqft", "median"),
        median_rate_per_sqft=("rate_per_sqft", "median")
    )
    .reindex(segment_order)
)

segment_summary["share_pct"] = (
    segment_summary["listings"]
    .div(len(df))
    .mul(100)
    .round(2)
)

segment_summary = segment_summary[
    [
        "listings",
        "share_pct",
        "median_price_lakh",
        "median_area_sqft",
        "median_rate_per_sqft"
    ]
]

print("Market segment summary:")
print(segment_summary.to_string())


# Dominant property type within each segment
segment_property_type = (
    df.groupby("market_segment", observed=True)["flat_type"]
    .agg(lambda x: x.value_counts().index[0])
    .reindex(segment_order)
)

print("\nDominant property type:")
print(segment_property_type.to_string())


# Dominant BHK group within each segment
segment_bhk = (
    df.groupby("market_segment", observed=True)["bhk_group"]
    .agg(lambda x: x.value_counts().index[0])
    .reindex(segment_order)
)

print("\nDominant BHK group:")
print(segment_bhk.to_string())


# Segment distribution by property type
segment_type_counts = pd.crosstab(
    df["market_segment"],
    df["flat_type"]
).reindex(segment_order)

print("\nProperty type counts by segment:")
print(segment_type_counts.to_string())

Market segment summary:
                listings  share_pct  median_price_lakh  median_area_sqft  median_rate_per_sqft
market_segment                                                                                
Budget              2149      15.11               60.0            1000.0                5353.0
Mid-Market          3343      23.50              150.0            1593.0                9111.0
Premium             5578      39.21              323.0            2208.0               14444.0
High-End            2315      16.27              654.0            3505.0               19000.0
Luxury               840       5.91             1375.0            6000.0               22000.0

Dominant property type:
market_segment
Budget        Apartment
Mid-Market    Apartment
Premium       Apartment
High-End      Apartment
Luxury        Apartment

Dominant BHK group:
market_segment
Budget        2
Mid-Market    3
Premium       3
High-End      4
Luxury        4

Property type counts by segment:
f